### Filtrado de Características y Preprocesamiento

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import gc

print("Iniciando pipeline de datos optimizado...")

# 1. Definición explícita de características validadas por SHAP
features_seleccionadas = [
    'iat', 'rst_count', 'urg_count', 'number', 'variance', 'tot_size', 
    'max', 'header_length', 'flow_duration', 'weight', 'rate', 'duration', 
    'protocol_type', 'syn_flag_number', 'fin_count', 'syn_count',
    'rst_flag_number', 'ack_count'
]

# CRÍTICO: Añadimos 'label' a la lista para poder hacer el filtrado y balanceo
columnas_a_leer = features_seleccionadas + ['label']

# 2. Carga eficiente desde Feather (Solo leemos lo que necesitamos)
print("Cargando dataset desde Feather...")
# Asegúrate de que el archivo 'dataset_eda_temp.feather' esté en tu directorio
df = pd.read_feather('dataset_eda_temp.feather', columns=columnas_a_leer)

# 3. Separar tráfico benigno y malicioso
df_benign = df[df['label'] == 'BenignTraffic']
df_malicious = df[df['label'] != 'BenignTraffic']

# Liberamos el dataframe original de la memoria RAM
del df
gc.collect()

n_benign = len(df_benign)
print(f"Cantidad de tráfico benigno: {n_benign}")

# 4. Muestreo estratificado del tráfico malicioso para balancear 1:1
# Mantiene las proporciones internas de los distintos tipos de ataques
_, df_malicious_sampled = train_test_split(
    df_malicious,
    test_size=n_benign,
    stratify=df_malicious['label'],
    random_state=42
)

print(f"Cantidad de tráfico malicioso muestreado: {len(df_malicious_sampled)}")

# Unir y mezclar (shuffle) el dataset balanceado
df_balanced = pd.concat([df_benign, df_malicious_sampled]).sample(frac=1, random_state=42).reset_index(drop=True)

# 5. Extraer matrices X e y para el NAS
X = df_balanced[features_seleccionadas].values
# Crear la etiqueta binaria: 0 = Benigno, 1 = Malicioso
y = (df_balanced['label'] != 'BenignTraffic').astype(int).values

# Limpieza profunda de RAM antes de arrancar TensorFlow
del df_benign, df_malicious, df_malicious_sampled, df_balanced
gc.collect()

print("Realizando divisiones de Train, Validation y Test...")
# 6. División de datos (70% Train, 15% Validation, 15% Test)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=42, stratify=y_temp)

# 7. Estandarización (CRÍTICO para punto flotante)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

input_dimension = X_train_scaled.shape[1]

print("="*50)
print(f"PREPROCESAMIENTO FINALIZADO")
print(f"Dimensión de entrada final: {input_dimension} características.")
print(f"Muestras de entrenamiento: {X_train_scaled.shape[0]}")
print(f"Muestras de validación: {X_val_scaled.shape[0]}")
print("="*50)

Iniciando pipeline de datos optimizado...
Cargando dataset desde Feather...
Cantidad de tráfico benigno: 1098195
Cantidad de tráfico malicioso muestreado: 1098195
Realizando divisiones de Train, Validation y Test...
PREPROCESAMIENTO FINALIZADO
Dimensión de entrada final: 18 características.
Muestras de entrenamiento: 1537417
Muestras de validación: 329514


### NAS Avanzado con Batch Normalization, Dropout y Pooling

In [5]:
import keras_tuner as kt
from tensorflow import keras
from tensorflow.keras import layers

def build_advanced_unified_model(hp):
    model = keras.Sequential()
    model.add(keras.Input(shape=(input_dimension,)))
    
    topology_type = hp.Choice('topology_type', ['mlp', 'cnn_1d'])
    
    if topology_type == 'mlp':
        # ==========================================
        # RUTA 1: PERCEPTRÓN MULTICAPA (MLP)
        # ==========================================
        for i in range(hp.Int('mlp_layers', 1, 4)):
            # Capa Densa
            model.add(layers.Dense(units=hp.Choice(f'mlp_units_{i}', values=[16, 32, 64])))
            
            # Búsqueda de Batch Normalization
            if hp.Boolean(f'mlp_batchnorm_{i}'):
                model.add(layers.BatchNormalization())
                
            model.add(layers.Activation('relu'))
            
            # Búsqueda de Dropout con tasa variable
            if hp.Boolean(f'mlp_dropout_{i}'):
                model.add(layers.Dropout(rate=hp.Float(f'mlp_dropout_rate_{i}', min_value=0.1, max_value=0.3, step=0.1)))
                
    else:
        # ==========================================
        # RUTA 2: RED CONVOLUCIONAL 1D (CNN)
        # ==========================================
        model.add(layers.Reshape((input_dimension, 1)))
        
        for i in range(hp.Int('cnn_layers', 1, 3)):
            # Capa Convolucional
            model.add(layers.Conv1D(
                filters=hp.Choice(f'cnn_filters_{i}', values=[8, 16, 32]),
                kernel_size=hp.Choice(f'cnn_kernel_{i}', values=[2, 3]),
                padding='same'
            ))
            
            # Búsqueda de Batch Normalization para CNN
            if hp.Boolean(f'cnn_batchnorm_{i}'):
                model.add(layers.BatchNormalization())
                
            model.add(layers.Activation('relu'))
            
            # Búsqueda de Pooling (Reducción de dimensionalidad)
            if hp.Boolean(f'cnn_pooling_{i}'):
                model.add(layers.MaxPooling1D(pool_size=2, padding='same'))
        
        # Flatten obligatorio para pasar de tensores 3D a 1D antes de la salida
        model.add(layers.Flatten())
        
        # Búsqueda de una capa Densa intermedia post-convolución
        if hp.Boolean('cnn_extra_dense'):
            model.add(layers.Dense(units=hp.Choice('cnn_dense_units', values=[16, 32])))
            model.add(layers.Activation('relu'))
            if hp.Boolean('cnn_dense_dropout'):
                model.add(layers.Dropout(rate=0.2))

    # ==========================================
    # CAPA DE SALIDA Y COMPILACIÓN
    # ==========================================
    model.add(layers.Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(name='auc')]
    )
    
    return model

print("Espacio de búsqueda avanzado definido con éxito.")

Espacio de búsqueda avanzado definido con éxito.


In [7]:
# Instanciar el buscador
tuner = kt.Hyperband(
    build_advanced_unified_model,
    objective=kt.Objective("val_auc", direction="max"),
    max_epochs=30,
    factor=3,
    directory='nas_logs',
    project_name='ids_unified_search_v1'
)

# Callbacks: Detener el entrenamiento si no mejora en 5 épocas
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=5, 
    restore_best_weights=True
)

print("Iniciando NAS en la RTX 3060. Esto puede tomar unos minutos...")

# Iniciar la búsqueda
tuner.search(
    X_train_scaled, y_train,
    epochs=30,
    validation_data=(X_val_scaled, y_val),
    callbacks=[early_stopping],
    batch_size=256 # Tamaño de lote grande para aprovechar la VRAM de la GPU
)

# --- RESULTADOS ---
print("\n" + "="*50)
print("BÚSQUEDA NAS FINALIZADA")
print("="*50)

# 1. Extraer los mejores hiperparámetros
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
ganador = best_hps.get('topology_type')

print(f"\nLa arquitectura ganadora es: **{ganador.upper()}**")
print(f"Tasa de aprendizaje óptima: {best_hps.get('learning_rate')}")

# 2. Reconstruir el modelo ganador
best_model = tuner.hypermodel.build(best_hps)

# 3. Mostrar resumen de la topología para tu Documento de Diseño
print("\nTopología exacta del modelo ganador:")
best_model.summary()

# (Opcional) Evaluar el modelo sin entrenar en test solo para verificar tensores
# Para el Hito 3, tu siguiente paso será entrenar este 'best_model' desde cero 
# con más épocas usando X_train_scaled.

Trial 90 Complete [00h 07m 52s]
val_auc: 0.998327910900116

Best val_auc So Far: 0.9987204074859619
Total elapsed time: 02h 32m 54s

BÚSQUEDA NAS FINALIZADA

La arquitectura ganadora es: **MLP**
Tasa de aprendizaje óptima: 0.001

Topología exacta del modelo ganador:
Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_2 (Dense)             (None, 32)                608       
                                                                 
 activation_3 (Activation)   (None, 32)                0         
                                                                 
 dense_3 (Dense)             (None, 64)                2112      
                                                                 
 batch_normalization_1 (Batc  (None, 64)               256       
 hNormalization)                                                 
                                                   